In [ ]:
import os
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("kj-zy1ya").project("crack-btlnb")
version = project.version(4)
dataset = version.download("yolov8")
                

# CRACK SEGMENTATION 

# Import libraries

In [ ]:
# ============================================================
# Cell 2: Import
# ============================================================
import os, yaml, random, shutil
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import TABLEAU_COLORS

##  Configuration

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================
DATA_DIR    = Path(r"D:\\Code\\Hu_hong_đường\\crack-4")
SPLITS      = ["train", "valid", "test"]
IMG_EXTS    = {".jpg", ".jpeg", ".png", ".bmp"}
COLORS      = list(TABLEAU_COLORS.values())

# Đọc data.yaml để lấy tên lớp
YAML_PATH = DATA_DIR / "data.yaml"

def load_yaml():
    with open(YAML_PATH, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

cfg = load_yaml()
CLASS_NAMES = cfg.get("names", [])
NC          = cfg.get("nc", len(CLASS_NAMES))
print(f"[INFO] Số lớp (nc): {NC}")
print(f"[INFO] Tên các lớp : {CLASS_NAMES}")


## Kiểm tra phân bố nhãn (label distribution)

In [ ]:
# ============================================================
# Cell 4: Kiểm tra phân bố nhãn (label distribution)
# ============================================================
def check_label_distribution():
    print("\n" + "="*60)
    print("KIỂM TRA 1: PHÂN BỐ NHÃN THEO TỪNG SPLIT")
    print("="*60)

    all_counts = {}
    for split in SPLITS:
        label_dir = DATA_DIR / split / "labels"
        if not label_dir.exists():
            print(f"[WARN] Không tìm thấy: {label_dir}")
            continue

        counter     = Counter()
        empty_files = []
        invalid_cls = []

        for lbl_file in label_dir.glob("*.txt"):
            lines = lbl_file.read_text(encoding="utf-8").strip().splitlines()
            if not lines:
                empty_files.append(lbl_file.name)
                continue
            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                if cls_id < 0 or cls_id >= NC:
                    invalid_cls.append((lbl_file.name, cls_id))
                counter[cls_id] += 1

        all_counts[split] = counter
        total = sum(counter.values())
        print(f"\n[{split.upper()}] Tổng polygon: {total} | File rỗng: {len(empty_files)}")
        for cid in sorted(counter):
            name = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else f"id_{cid}"
            pct  = counter[cid] / total * 100 if total else 0
            print(f"  Lớp {cid:2d} ({name:20s}): {counter[cid]:6d} ({pct:5.1f}%)")
        if empty_files:
            print(f"  [CẢNH BÁO] File rỗng ({len(empty_files)}): {empty_files[:5]}")
        if invalid_cls:
            print(f"  [LỖI] Class ID ngoài [0,{NC-1}]: {invalid_cls[:5]}")

    # Biểu đồ
    if not all_counts:
        return
    fig, axes = plt.subplots(1, len(all_counts), figsize=(6*len(all_counts), 5))
    if len(all_counts) == 1:
        axes = [axes]
    for ax, (split, counter) in zip(axes, all_counts.items()):
        labels = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"id_{i}" for i in sorted(counter)]
        values = [counter[i] for i in sorted(counter)]
        bars = ax.bar(labels, values, color=COLORS[:len(labels)])
        ax.set_title(f"{split} — phân bố nhãn", fontsize=12, fontweight="bold")
        ax.set_ylabel("Số lượng polygon")
        ax.tick_params(axis="x", rotation=30)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(values)*0.01,
                    str(v), ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    out = DATA_DIR / "label_distribution.png"
    plt.savefig(str(out), dpi=120)
    plt.show()
    print(f"\n[SAVED] {out}")

check_label_distribution()

## Kiểm tra label có nằm đúng trong ảnh không

In [ ]:
# ============================================================
# Cell 5: Kiểm tra polygon có nằm trong ảnh không
# YOLO-Seg format: class x1 y1 x2 y2 ... xn yn
# Tất cả tọa độ phải trong [0, 1]
# ============================================================
def check_polygon_bounds():
    print("\n" + "="*60)
    print("KIỂM TRA 2: POLYGON CÓ NẰM TRONG ẢNH KHÔNG")
    print("="*60)

    TOL = 1e-4
    issues = defaultdict(list)

    for split in SPLITS:
        label_dir = DATA_DIR / split / "labels"
        if not label_dir.exists():
            continue

        for lbl_file in label_dir.glob("*.txt"):
            for ln, line in enumerate(
                    lbl_file.read_text(encoding="utf-8").strip().splitlines(), 1):
                parts = line.strip().split()
                # YOLO-Seg: 1 class + ít nhất 3 điểm (6 tọa độ) = 7 phần
                if len(parts) < 7:
                    continue
                coords = list(map(float, parts[1:]))
                if len(coords) % 2 != 0:
                    issues[split].append(
                        f"{lbl_file.name} ln{ln}: số tọa độ lẻ ({len(coords)})")
                    continue
                xs = coords[0::2]
                ys = coords[1::2]
                bad_x = [x for x in xs if x < -TOL or x > 1 + TOL]
                bad_y = [y for y in ys if y < -TOL or y > 1 + TOL]
                if bad_x:
                    issues[split].append(
                        f"{lbl_file.name} ln{ln}: x ngoài [0,1] → {bad_x[:3]}")
                if bad_y:
                    issues[split].append(
                        f"{lbl_file.name} ln{ln}: y ngoài [0,1] → {bad_y[:3]}")

        n = len(issues.get(split, []))
        status = "✅ OK" if n == 0 else f"❌ {n} vấn đề"
        print(f"[{split.upper():6s}] {status}")
        for msg in issues.get(split, [])[:10]:
            print(f"  → {msg}")

    return issues

poly_issues = check_polygon_bounds()

## Kiểm tra kích thước ảnh

In [ ]:
# ============================================================
# Cell 6: Kiểm tra kích thước ảnh
# ============================================================
def check_image_sizes():
    print("\n" + "="*60)
    print("KIỂM TRA 3: KÍCH THƯỚC ẢNH")
    print("="*60)

    for split in SPLITS:
        img_dir = DATA_DIR / split / "images"
        if not img_dir.exists():
            continue

        size_counter = Counter()
        corrupt      = []

        for img_path in img_dir.iterdir():
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                corrupt.append(img_path.name)
                continue
            h, w = img.shape[:2]
            size_counter[(w, h)] += 1

        total = sum(size_counter.values())
        print(f"\n[{split.upper()}] Ảnh hợp lệ: {total} | Ảnh hỏng: {len(corrupt)}")
        for (w, h), cnt in size_counter.most_common(5):
            print(f"  {w}x{h} — {cnt} ảnh ({cnt/total*100:.1f}%)")
        if len(size_counter) > 1:
            print(f"  [CẢNH BÁO] Có {len(size_counter)} kích thước khác nhau!")
        if corrupt:
            print(f"  [LỖI] Ảnh hỏng: {corrupt[:5]}")

check_image_sizes()

In [ ]:
# ============================================================
# Cell 7: Hiển thị polygon segmentation trên ảnh
# ============================================================
def visualize_seg_labels(split="train", n_samples=9, seed=42):
    print("\n" + "="*60)
    print(f"KIỂM TRA 4: HIỂN THỊ POLYGON TRÊN ẢNH [{split}]")
    print("="*60)

    img_dir   = DATA_DIR / split / "images"
    label_dir = DATA_DIR / split / "labels"
    img_files = [p for p in img_dir.iterdir()
                 if p.suffix.lower() in IMG_EXTS]

    random.seed(seed)
    samples = random.sample(img_files, min(n_samples, len(img_files)))

    cols = 3
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, samples):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = label_dir / (img_path.stem + ".txt")
        overlay  = img.copy()

        if lbl_path.exists():
            for line in lbl_path.read_text(encoding="utf-8").strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 7:
                    continue
                cls_id = int(parts[0])
                coords = list(map(float, parts[1:]))
                if len(coords) % 2 != 0:
                    continue
                xs = [x * w for x in coords[0::2]]
                ys = [y * h for y in coords[1::2]]
                pts = np.array(list(zip(xs, ys)), dtype=np.int32)

                color = tuple(int(c * 255) for c in
                              plt.cm.tab10(cls_id % 10)[:3])
                # Vẽ polygon fill bán trong suốt
                cv2.fillPoly(overlay, [pts], color)
                # Vẽ đường viền
                cv2.polylines(img, [pts], isClosed=True,
                              color=color, thickness=2)
                # Label text
                name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) \
                       else f"id_{cls_id}"
                cv2.putText(img, name, (pts[0][0], max(pts[0][1]-5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1,
                            cv2.LINE_AA)

        # Blend overlay
        img = cv2.addWeighted(img, 0.7, overlay, 0.3, 0)
        ax.imshow(img)
        ax.set_title(img_path.name[:30], fontsize=8)
        ax.axis("off")

    for ax in axes[len(samples):]:
        ax.axis("off")

    plt.tight_layout()
    out = DATA_DIR / f"seg_viz_{split}.png"
    plt.savefig(str(out), dpi=120)
    plt.show()
    print(f"[SAVED] {out}")

visualize_seg_labels(split="train", n_samples=9)

In [ ]:
# ============================================================
# Cell 8: Kiểm tra ghép cặp ảnh — label
# ============================================================
def check_image_label_pairing():
    print("\n" + "="*60)
    print("KIỂM TRA 5: KHỚP GIỮA ẢNH VÀ FILE NHÃN")
    print("="*60)

    for split in SPLITS:
        img_dir   = DATA_DIR / split / "images"
        label_dir = DATA_DIR / split / "labels"
        if not img_dir.exists():
            continue

        img_stems   = {p.stem for p in img_dir.iterdir()
                       if p.suffix.lower() in IMG_EXTS}
        label_stems = ({p.stem for p in label_dir.glob("*.txt")}
                       if label_dir.exists() else set())

        no_label = img_stems - label_stems
        no_img   = label_stems - img_stems
        paired   = img_stems & label_stems

        print(f"\n[{split.upper():6s}]  Ảnh: {len(img_stems)}"
              f"  |  Label: {len(label_stems)}")
        print(f"   ✅ Khớp đôi    : {len(paired)}")
        if no_label:
            print(f"   ❌ Ảnh thiếu label ({len(no_label)}): "
                  f"{list(no_label)[:5]}")
        if no_img:
            print(f"   ❌ Label thừa ({len(no_img)}): "
                  f"{list(no_img)[:5]}")

check_image_label_pairing()


## XÓA FILE LABEL RỖNG

In [ ]:
# ============================================================
# Cell 9: XÓA FILE LABEL RỖNG
# ============================================================
def delete_empty_labels():
    print("="*60)
    print("XÓA FILE LABEL RỖNG")
    print("="*60)
    
    deleted = {}
    for split in SPLITS:
        label_dir = DATA_DIR / split / "labels"
        if not label_dir.exists():
            continue
            
        empty_count = 0
        for lbl_file in label_dir.glob("*.txt"):
            content = lbl_file.read_text(encoding="utf-8").strip()
            if not content or content == "":
                lbl_file.unlink()
                empty_count += 1
        
        deleted[split] = empty_count
        print(f"[{split.upper():6s}] Xóa {empty_count} file rỗng")
    
    total_deleted = sum(deleted.values())
    print(f"\n✅ Tổng cộng xóa {total_deleted} file label rỗng")
    print("Chạy lại Cell 4 để xác nhận!")

delete_empty_labels()

In [ ]:

import time
from datetime import datetime
from ultralytics import YOLO
from pathlib import Path

DATA_DIR = Path(r"D:\\Code\\Hu_hong_đường\\crack-4")
YAML_PATH = DATA_DIR / "data.yaml"

def log_with_time(log_path, msg):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f"[{timestamp}] {msg}"
    print(line)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(line + "\\n")

def train_model(model_name, name_suffix, batch_size=8):
    log_path = DATA_DIR / "runs" / f"seg_train_log_{name_suffix}.txt"
    
    start_dt = datetime.now()
    start_ts = time.time()
    
    def log(msg):
        log_with_time(log_path, msg)
    
    log(f"=== TRAIN {model_name.upper()} ===")
    log(f"Data: {YAML_PATH}")
    log(f"Epochs: 100 | Imgsz: 640 | Batch: {batch_size}")
    log(f"Bắt đầu: {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")
    
    model = YOLO(model_name)
    results = model.train(
        data=str(YAML_PATH),
        epochs=100,
        imgsz=640,
        batch=batch_size,
        patience=20,
        device=0,
        project=str(DATA_DIR / "runs"),
        name=f"crack_seg_{name_suffix}",
        exist_ok=True,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=5, translate=0.1, scale=0.5,
        flipud=0.0, fliplr=0.5, mosaic=1.0,
    )
    
    elapsed = time.time() - start_ts
    hh, mm, ss = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    
    log(f"=== XONG {model_name.upper()} ===")
    log(f"Thời gian: {hh:02d}h {mm:02d}m {ss:02d}s")
    if hasattr(results, "results_dict"):
        for k, v in results.results_dict.items():
            if isinstance(v, float):
                log(f"{k:<30s}: {v:.4f}")
    log(f"Best: runs/crack_seg_{name_suffix}/weights/best.pt")
    
    return results

In [ ]:

# Cell 10: Train YOLOv8n-seg (nhỏ nhất, nhanh nhất)
print("=== CELL 10: YOLOv8n-seg ===")
train_model("yolov8n-seg.pt", "nano")

In [ ]:

# Cell 10: Train YOLOv8n-seg (nhỏ nhất, nhanh nhất)
print("=== CELL 10: YOLOv8n-seg ===")
# train_model("yolov8n-seg.pt", "nano")

# Cell 11: Train YOLOv8m-seg (cân bằng)
print("\\n=== CELL 11: YOLOv8m-seg ===")
# train_model("yolov8m-seg.pt", "medium", batch_size=4)

# Cell 12: Train YOLOv8s-seg (nhanh vừa)
print("\\n=== CELL 12: YOLOv8s-seg ===")
# train_model("yolov8s-seg.pt", "small", batch_size=8)

print("\\n✅ Thêm comment để chạy từng model. Bắt đầu từ nano!")